# CBIS-DDSM: Lesion Segmentation with an **Attention U-Net** (supervised)

After the weakly-supervised attention experiments failed to localize (pointing-game ~0-5%), this
uses the ROI masks **directly** as supervision -> real localization numbers.

- **Input:** full mammogram (CLAHE, 256x256).  **Target:** union of the lesion ROI masks.
- **Model:** **Attention U-Net** (Oktay et al. 2018) - attention gates on the skip connections
  suppress irrelevant background and focus on the lesion (the attention is now *trained with the
  mask*, not hoped-for from a classifier).
- **Loss:** BCE + Dice.  **Metrics:** Dice, IoU (patient-disjoint official test).

Keeps the infra fixes: RTX pinned (hide Intel iGPU), uint8 image/mask arrays, patient-disjoint.


In [ ]:
# ---- GPU: pin to RTX ----
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if gpus:
    try:
        tf.config.set_visible_devices([gpus[0]], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e:
        print('gpu cfg:', e)
print('Visible:', tf.config.get_visible_devices('GPU'))

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# ============================================================ CONFIG
DATA_ROOT  = 'D:/cbis-ddsm'; JPEG_ROOT = DATA_ROOT + '/jpeg'; CSV_DIR = DATA_ROOT + '/csv'
DICOM_INFO = CSV_DIR + '/dicom_info.csv'; SAVE_DIR = 'G:/My Drive/VS_CC/models'
os.makedirs(SAVE_DIR, exist_ok=True)
TRAIN_CSVS = ['mass_case_description_train_set.csv', 'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',  'calc_case_description_test_set.csv']
SZ = 256; BASE = 32; BATCH = 4; EPOCHS = 60; RANDOM_STATE = 2021; MASK_THRESH = 127
IMG_SIZE = SZ

## 1. Resolve full image + ROI mask (dicom_info.csv)

In [ ]:
def _to_local(p):
    p = str(p).replace('\\', '/'); i = p.find('jpeg/')
    return os.path.join(JPEG_ROOT, p[i + 5:]) if i >= 0 else p
def build_uid_maps(dinfo):
    di = pd.read_csv(dinfo); di['SeriesDescription'] = di['SeriesDescription'].fillna('')
    di['local'] = di['image_path'].apply(_to_local); di['uid'] = di['SeriesInstanceUID'].astype(str)
    full, mask, allf = {}, {}, {}
    for _, r in di.iterrows():
        allf.setdefault(r['uid'], []).append(r['local']); d = r['SeriesDescription'].lower()
        if 'full' in d: full.setdefault(r['uid'], r['local'])
        elif 'roi mask' in d: mask.setdefault(r['uid'], r['local'])
    return full, mask, allf
FULL_MAP, MASK_MAP, ANY_MAP = build_uid_maps(DICOM_INFO)
def series_uid(p):
    q = str(p).replace('\\', '/').strip().split('/'); return q[-2] if len(q) >= 2 else None
def is_mask_like(g):
    return g is not None and np.mean((g <= 10) | (g >= 245)) > 0.90 and len(np.unique(g)) <= 64
def resolve_mask(p):
    u = series_uid(p)
    if u in MASK_MAP: return MASK_MAP[u]
    for c in ANY_MAP.get(u, []):
        if is_mask_like(cv2.imread(c, cv2.IMREAD_GRAYSCALE)): return c
    return None
def load_tables(names):
    df = pd.concat([pd.read_csv(os.path.join(CSV_DIR, n)) for n in names if os.path.exists(os.path.join(CSV_DIR, n))], ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]; return df
def prep(df):
    df['full_path'] = df['image_file_path'].apply(lambda p: FULL_MAP.get(series_uid(p)))
    df['mask_path'] = df['roi_mask_file_path'].apply(resolve_mask)
    df['patient_id'] = df['patient_id'].astype(str); df['image_key'] = df['image_file_path'].astype(str)
    return df[['patient_id', 'image_key', 'full_path', 'mask_path']].dropna(subset=['full_path'])
train_df = prep(load_tables(TRAIN_CSVS)); test_df = prep(load_tables(TEST_CSVS))
ov = set(train_df.patient_id) & set(test_df.patient_id)
if ov: train_df = train_df[~train_df.patient_id.isin(ov)].reset_index(drop=True)
print('Train rows', len(train_df), 'Test rows', len(test_df), 'disjoint', set(train_df.patient_id).isdisjoint(set(test_df.patient_id)))

## 2. Data: disk generator for train (RAM-safe), small test set in arrays

The machine is RAM-tight, so training streams image+union-mask batches from disk via a
`Sequence` (only a batch is ever in RAM). The test set is small, so we load it into arrays for
per-image Dice/IoU + overlays.

In [ ]:
_clahe = cv2.createCLAHE(2.0, (8, 8))
def load_img(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    return np.stack([_clahe.apply(cv2.resize(img, (SZ, SZ)))] * 3, -1)   # uint8 [0,255]
def load_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) if path else None
    if not is_mask_like(m): return None
    return (cv2.resize(m, (SZ, SZ), interpolation=cv2.INTER_NEAREST) >= MASK_THRESH).astype(np.uint8)
def union_mask(mask_paths):
    mask = np.zeros((SZ, SZ), np.uint8)
    for mp in mask_paths:
        u = load_mask(mp)
        if u is not None: mask |= u
    return mask

def make_items(df):
    d = df[df['mask_path'].notna()]
    items, pats = [], []
    for key, rows in d.groupby('image_key'):
        items.append((rows.iloc[0]['full_path'], list(rows['mask_path'])))
        pats.append(rows.iloc[0]['patient_id'])
    return items, np.array(pats)

items_tr, pats_tr = make_items(train_df)
items_te, pats_te = make_items(test_df)
print('masked mammograms -> train', len(items_tr), '| test', len(items_te))

# ---- decode-once memmap cache (avoids re-decoding JPEGs every epoch; RAM-safe) ----
CACHE = 'D:/cbis-ddsm/_seg_cache'; os.makedirs(CACHE, exist_ok=True)
def _p(prefix, ch): return os.path.join(CACHE, f'{prefix}_{ch}_{SZ}.npy')
def build_cache(items, prefix):
    if os.path.exists(_p(prefix, 'X')) and os.path.exists(_p(prefix, 'Y')):
        print(f'  cache {prefix} exists -> skip'); return
    n = len(items)
    X = np.lib.format.open_memmap(_p(prefix, 'X'), mode='w+', dtype=np.uint8, shape=(n, SZ, SZ, 3))
    Y = np.lib.format.open_memmap(_p(prefix, 'Y'), mode='w+', dtype=np.uint8, shape=(n, SZ, SZ, 1))
    for i, (fp, mps) in enumerate(items):
        img = load_img(fp)
        if img is not None: X[i] = img
        Y[i] = union_mask(mps)[..., None]
        if (i + 1) % 400 == 0: print('   decoded', i + 1)
    X.flush(); Y.flush(); del X, Y
print('caching TRAIN...'); build_cache(items_tr, 'tr')
print('caching TEST...');  build_cache(items_te, 'te')
Xtr = np.load(_p('tr', 'X'), mmap_mode='r'); Ytr = np.load(_p('tr', 'Y'), mmap_mode='r')
Xte = np.load(_p('te', 'X'), mmap_mode='r'); Yte = np.load(_p('te', 'Y'), mmap_mode='r')
print('cached memmaps -> Xtr', Xtr.shape, '| Xte', Xte.shape, '| test coverage %.4f' % (np.asarray(Yte[:50]).mean()))

class SegSeq(tf.keras.utils.Sequence):
    # reads batches from the memmap cache (fast; only a batch is resident)
    def __init__(self, X, Y, idx, batch, shuffle=True):
        self.X = X; self.Y = Y; self.idx = np.asarray(idx); self.batch = batch; self.shuffle = shuffle; self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.idx) / self.batch))
    def on_epoch_end(self):
        self.order = np.arange(len(self.idx))
        if self.shuffle: np.random.shuffle(self.order)
    def __getitem__(self, b):
        sel = np.sort(self.idx[self.order[b*self.batch:(b+1)*self.batch]])
        return self.X[sel].astype('float32'), self.Y[sel].astype('float32')

### 2b. DIAGNOSTIC — image + union mask alignment

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2), dpi=110)
for a, idx in zip(ax, np.random.RandomState(0).choice(len(Xte), 4, replace=False)):
    a.imshow(Xte[idx][..., 0], cmap='gray'); a.contour(Yte[idx, :, :, 0], colors='lime', linewidths=1.2)
    a.axis('off')
plt.tight_layout(); plt.show()

## 3. Attention U-Net (Oktay-style attention gates on skips)

In [ ]:
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    return x
def attn_gate(x, g, inter):
    tx = layers.Conv2D(inter, 1)(x); pg = layers.Conv2D(inter, 1)(g)
    r = layers.Activation('relu')(layers.Add()([tx, pg]))
    psi = layers.Conv2D(1, 1, activation='sigmoid')(r)
    return layers.Multiply()([x, psi])

def build_att_unet(sz=SZ, base=BASE):
    inp = layers.Input((sz, sz, 3)); s = layers.Rescaling(1./255)(inp)
    c1 = conv_block(s, base);     p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, base*2);  p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, base*4);  p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, base*8);  p4 = layers.MaxPooling2D()(c4)
    bn = conv_block(p4, base*16)
    u4 = layers.UpSampling2D()(bn); a4 = attn_gate(c4, u4, base*8); c4d = conv_block(layers.Concatenate()([u4, a4]), base*8)
    u3 = layers.UpSampling2D()(c4d); a3 = attn_gate(c3, u3, base*4); c3d = conv_block(layers.Concatenate()([u3, a3]), base*4)
    u2 = layers.UpSampling2D()(c3d); a2 = attn_gate(c2, u2, base*2); c2d = conv_block(layers.Concatenate()([u2, a2]), base*2)
    u1 = layers.UpSampling2D()(c2d); a1 = attn_gate(c1, u1, base);   c1d = conv_block(layers.Concatenate()([u1, a1]), base)
    out = layers.Conv2D(1, 1, activation='sigmoid')(c1d)
    return Model(inp, out)

## 4. Dice/BCE loss + metrics, compile

In [ ]:
def dice_coef(y_true, y_pred, smooth=1.):
    yt = tf.reshape(tf.cast(y_true, tf.float32), [-1]); yp = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(yt * yp)
    return (2*inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)
def dice_loss(y_true, y_pred): return 1 - dice_coef(y_true, y_pred)
def bce_dice(y_true, y_pred):
    yt = tf.cast(y_true, tf.float32)
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(yt, y_pred)) + dice_loss(yt, y_pred)
def iou_metric(y_true, y_pred, smooth=1.):
    yt = tf.reshape(tf.cast(y_true, tf.float32), [-1]); yp = tf.reshape(tf.cast(y_pred > 0.5, tf.float32), [-1])
    inter = tf.reduce_sum(yt * yp); union = tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter
    return (inter + smooth) / (union + smooth)

model = build_att_unet()
model.compile(Adam(1e-3), loss=bce_dice, metrics=[dice_coef, iou_metric])
print('params: %.2fM' % (model.count_params()/1e6))

## 5. Train (patient-disjoint val split)

In [ ]:
gss = GroupShuffleSplit(1, test_size=0.15, random_state=RANDOM_STATE)
tr, va = next(gss.split(np.arange(len(items_tr)), groups=pats_tr))
train_gen = SegSeq(Xtr, Ytr, tr, BATCH, shuffle=True)
val_gen   = SegSeq(Xtr, Ytr, va, BATCH, shuffle=False)
print('fit', len(tr), 'val', len(va))
hist = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                 callbacks=[EarlyStopping(monitor='val_dice_coef', mode='max', patience=12, restore_best_weights=True),
                            ReduceLROnPlateau(monitor='val_dice_coef', mode='max', factor=0.5, patience=5)], verbose=2)

## 6. Evaluate on the patient-disjoint test set (per-image Dice / IoU)

In [ ]:
P = model.predict(Xte, batch_size=BATCH, verbose=0)[..., 0]
def per_image(y, p, thr=0.5):
    dices, ious, hits = [], [], []
    for i in range(len(y)):
        gt = y[i, :, :, 0].astype(bool); pm = p[i] >= thr
        inter = np.logical_and(gt, pm).sum(); s = gt.sum() + pm.sum(); u = np.logical_or(gt, pm).sum()
        dices.append(2*inter/s if s else 1.0); ious.append(inter/u if u else 1.0)
        yy, xx = np.unravel_index(int(np.argmax(p[i])), p[i].shape); hits.append(int(gt[yy, xx]))
    return np.array(dices), np.array(ious), np.array(hits)
d, io, hit = per_image(Yte, P)
print(f'TEST  Dice {d.mean():.3f} +/- {d.std():.3f} | IoU {io.mean():.3f} +/- {io.std():.3f} '
      f'| pointing-game {hit.mean()*100:.1f}%  (n={len(d)})')

### 6b. Overlays: image + GT (green) + prediction (red)

In [ ]:
order = np.argsort(-d)[:6]
fig, ax = plt.subplots(1, 6, figsize=(18, 3.2), dpi=110)
for a, idx in zip(ax, order):
    a.imshow(Xte[idx][..., 0], cmap='gray')
    a.contour(Yte[idx, :, :, 0], colors='lime', linewidths=1.2)
    a.contour((P[idx] >= 0.5).astype(float), colors='red', linewidths=1.0)
    a.set_title(f'Dice {d[idx]:.2f}', fontsize=9); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'cbis_unet_overlays.png'), dpi=200, bbox_inches='tight'); plt.show()

## 7. Summary + save

In [ ]:
summary = (
    f"Attention U-Net (Oktay-style attention gates, base={BASE}, {SZ}x{SZ}) for supervised lesion "
    f"segmentation on CBIS-DDSM full mammograms (patient-disjoint official split; trained with "
    f"BCE+Dice on the ROI masks). Test: Dice {d.mean():.3f} +/- {d.std():.3f}, IoU {io.mean():.3f}, "
    f"pointing-game {hit.mean()*100:.1f}% over {len(d)} masked test mammograms. This supervised "
    f"result contrasts with the weakly-supervised attention baselines (pointing-game ~0-5%)."
)
print(summary)
open(os.path.join(SAVE_DIR, 'cbis_unet_summary.txt'), 'w').write(summary)
model.save_weights(os.path.join(SAVE_DIR, 'cbis_attention_unet.weights.h5'))
print('\nSaved to', SAVE_DIR)